# Fashion-MNIST 10 Binary Specialists & Multi-Task Generalist with Pareto Optimization (KD)

Notebook ini telah diperbaiki agar variabel `K_MAX = 4` digunakan secara konsisten pada penciptaan `gen_prefs` (4 preferensi slot tumpukan) dan pelatihan `MTLNet`.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, matplotlib.pyplot as plt, random, time, copy
from torchvision import datasets, models
import pandas as pd

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', dev)

CFG = dict(
    N_TRAIN_MULTI = 20000,
    N_VAL_MULTI   = 2000,
    N_TEST_MULTI  = 5000,
    N_SPEC_HARD   = 20000,
    LABELED_FRAC  = 0.05,
    BS            = 128,
    LR            = 1e-3,
    SPEC_EPOCHS   = 3,
    GEN_EPOCHS    = 4,
    T_KD          = 2.0,
    LAM           = 0.7,
    METHODS       = ['ls', 'pcgrad', 'mgda', 'famo'],
    N_PREF_EXTRA  = 2,
    EXPAND_ROUNDS = 1,
    SEED          = 0,
)

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed(CFG['SEED'])

In [ ]:
CANVAS, DS = 36, 28
K_MAX = 4

CLASS_NAMES = (
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
)
N_CLASSES = len(CLASS_NAMES)

fashion_tr = datasets.FashionMNIST('./data', train=True, download=True)
fashion_te = datasets.FashionMNIST('./data', train=False, download=True)
all_tr_x, all_tr_y = fashion_tr.data.numpy(), fashion_tr.targets.numpy()
fa_te_x, fa_te_y = fashion_te.data.numpy(), fashion_te.targets.numpy()

split_rng = np.random.RandomState(CFG['SEED'])
perm = split_rng.permutation(len(all_tr_x))
train_ids, val_ids = perm[:54000], perm[54000:]
fa_tr_x, fa_tr_y = all_tr_x[train_ids], all_tr_y[train_ids]
fa_val_x, fa_val_y = all_tr_x[val_ids], all_tr_y[val_ids]

def make_multi_flexible(imgs, labels, n, k_min=1, k_max=K_MAX, seed=1):
    rng = np.random.RandomState(seed)
    X = np.zeros((n, CANVAS, CANVAS), dtype=np.uint8)
    Y = np.full((n, k_max), -1, dtype=np.int64)
    P = np.full((n, k_max, 2), -1, dtype=np.int64)
    max_offset = CANVAS - DS

    for i in range(n):
        k_items = rng.randint(k_min, k_max + 1)
        idx = rng.randint(0, len(imgs), size=k_items)
        for k, j in enumerate(idx):
            pos_y = rng.randint(0, max_offset + 1)
            pos_x = rng.randint(0, max_offset + 1)
            patch = X[i, pos_y:pos_y+DS, pos_x:pos_x+DS].astype(np.uint16) + imgs[j].astype(np.uint16)
            X[i, pos_y:pos_y+DS, pos_x:pos_x+DS] = np.minimum(patch, 255).astype(np.uint8)
            Y[i, k] = labels[j]
            P[i, k] = [pos_y, pos_x]
            
    return X, Y, P

def crop_pad(X, P, k):
    out = np.zeros_like(X)
    off = (CANVAS - DS) // 2
    for i in range(len(X)):
        if P[i, k, 0] != -1:
            y, x = P[i, k]
            out[i, off:off+DS, off:off+DS] = X[i, y:y+DS, x:x+DS]
    return out

def pad_fashion(imgs):
    out = np.zeros((len(imgs), CANVAS, CANVAS), np.uint8)
    off = (CANVAS - DS) // 2
    out[:, off:off+DS, off:off+DS] = imgs
    return out

Xg_tr, Yg_tr, Pg_tr = make_multi_flexible(fa_tr_x, fa_tr_y, CFG['N_TRAIN_MULTI'], k_min=2, k_max=K_MAX, seed=1)
Xg_val, Yg_val, Pg_val = make_multi_flexible(fa_val_x, fa_val_y, CFG['N_VAL_MULTI'], k_min=2, k_max=K_MAX, seed=2)
Xg_te, Yg_te, Pg_te   = make_multi_flexible(fa_te_x, fa_te_y, CFG['N_TEST_MULTI'], k_min=2, k_max=K_MAX, seed=3)

Xs_c, Ys_c = pad_fashion(fa_tr_x), fa_tr_y
n_h = (CFG['N_SPEC_HARD'] + K_MAX - 1) // K_MAX
Xh_raw, Yh_raw, Ph_raw = make_multi_flexible(fa_tr_x, fa_tr_y, n_h, k_min=2, k_max=K_MAX, seed=4)
Xs_h = np.concatenate([crop_pad(Xh_raw, Ph_raw, k) for k in range(K_MAX)])[:CFG['N_SPEC_HARD']]
Ys_h = np.concatenate([Yh_raw[:, k] for k in range(K_MAX)])[:CFG['N_SPEC_HARD']]
nS = min(len(Xs_c), len(Xs_h))
Xs_c, Ys_c, Xs_h, Ys_h = Xs_c[:nS], Ys_c[:nS], Xs_h[:nS], Ys_h[:nS]

Xs_c_val, Ys_c_val = pad_fashion(fa_val_x), fa_val_y
Xs_h_val = np.concatenate([crop_pad(Xg_val, Pg_val, k) for k in range(K_MAX)])
Ys_h_val = np.concatenate([Yg_val[:, k] for k in range(K_MAX)])

Xs_c_te, Ys_c_te = pad_fashion(fa_te_x), fa_te_y
Xs_h_te = np.concatenate([crop_pad(Xg_te, Pg_te, k) for k in range(K_MAX)])
Ys_h_te = np.concatenate([Yg_te[:, k] for k in range(K_MAX)])

def as_dev(a):
    return torch.from_numpy(np.ascontiguousarray(a)).to(dev)

Xg_tr_t, Yg_tr_t = as_dev(Xg_tr), as_dev(Yg_tr)
Xg_val_t, Yg_val_t = as_dev(Xg_val), as_dev(Yg_val)
Xg_te_t, Yg_te_t = as_dev(Xg_te), as_dev(Yg_te)

Xs_c_t, Ys_c_t, Xs_h_t, Ys_h_t = map(as_dev, (Xs_c, Ys_c, Xs_h, Ys_h))
Xs_c_val_t, Ys_c_val_t, Xs_h_val_t, Ys_h_val_t = map(as_dev, (Xs_c_val, Ys_c_val, Xs_h_val, Ys_h_val))
Xs_c_te_t, Ys_c_te_t, Xs_h_te_t, Ys_h_te_t = map(as_dev, (Xs_c_te, Ys_c_te, Xs_h_te, Ys_h_te))

spec_g = torch.Generator().manual_seed(CFG['SEED'])
spec_lab_mask = (torch.rand(len(Xs_c_t), generator=spec_g) < CFG['LABELED_FRAC']).to(dev)

In [ ]:
class BinarySpecialistNet(nn.Module):
    def __init__(self, arch='resnet18'):
        super().__init__()
        bb = getattr(models, arch)(weights=None)
        bb.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        bb.maxpool = nn.Identity()
        feat = bb.fc.in_features
        bb.fc = nn.Linear(feat, 1)
        self.backbone = bb
        
    def forward(self, x):
        return self.backbone(x).squeeze(-1)

class MTLNet(nn.Module):
    def __init__(self, arch, n_heads=K_MAX, n_cls=10):
        super().__init__()
        bb = getattr(models, arch)(weights=None)
        bb.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        bb.maxpool = nn.Identity()
        feat = bb.fc.in_features
        bb.fc = nn.Identity()
        self.backbone = bb
        self.heads = nn.ModuleList([nn.Linear(feat, n_cls) for _ in range(n_heads)])

    def forward(self, x):
        f = self.backbone(x)
        return [head(f) for head in self.heads]

def prep(xb, shift=0):
    x = xb.float().div(255).unsqueeze(1)
    if shift:
        sy = random.randint(-shift, shift)
        sx = random.randint(-shift, shift)
        x = F.pad(x, (shift, shift, shift, shift))
        x = x[:, :, shift + sy:shift + sy + CANVAS, shift + sx:shift + sx + CANVAS]
    return x

@torch.no_grad()
def predict_binary(model, X, bs=512, temp=1.0):
    model.eval()
    probs = []
    for s in range(0, len(X), bs):
        logits = model(prep(X[s:s+bs]))
        p = torch.sigmoid(logits / temp).cpu()
        probs.append(p)
    return torch.cat(probs)

@torch.no_grad()
def predict_gen(model, X, bs=512, temp=1.0):
    model.eval()
    outs = None
    for s in range(0, len(X), bs):
        o = model(prep(X[s:s+bs]))
        o = [F.softmax(z / temp, dim=-1).cpu() for z in o]
        outs = [[a] for a in o] if outs is None else [p + [a] for p, a in zip(outs, o)]
    return [torch.cat(p) for p in outs]

In [ ]:
def flat_grads(losses, params):
    gs = []
    for l in losses:
        g = torch.autograd.grad(l, params, retain_graph=True, allow_unused=True)
        gs.append(torch.cat([(gi if gi is not None else torch.zeros_like(p)).flatten() for gi, p in zip(g, params)]))
    return torch.stack(gs)

def pcgrad(G):
    T = len(G)
    out = G.clone()
    for i in range(T):
        for j in random.sample(range(T), T):
            if j == i: continue
            dot = (out[i] * G[j]).sum()
            if dot < 0:
                out[i] = out[i] - dot / (G[j].norm() ** 2 + 1e-12) * G[j]
    return out.sum(0)

def mgda_weights(G, iters=50):
    M = G @ G.T
    T = len(G)
    a = torch.ones(T, device=G.device) / T
    for _ in range(iters):
        i = torch.argmin(M @ a)
        e = torch.zeros_like(a)
        e[i] = 1
        d = e - a
        gamma = torch.clamp(-(d @ M @ a) / (d @ M @ d + 1e-12), 0, 1)
        a = a + gamma * d
    return a

class FAMO:
    def __init__(self, T, pref=None, lr=0.025, gamma=1e-3):
        init = torch.log(torch.tensor(pref if pref is not None else [1.0 / T] * T, dtype=torch.float32))
        self.w = init.clone().to(dev).requires_grad_(True)
        self.opt = torch.optim.Adam([self.w], lr=lr, weight_decay=gamma)
        self.eps = 1e-8
        self.min_loss = torch.zeros(T, device=dev)
    def weighted(self, losses):
        D = losses - self.min_loss + self.eps
        z = F.softmax(self.w, -1)
        c = (z / D).sum().detach()
        return (D.log() * z / c).sum()
    def update(self, prev, curr):
        delta = (prev - self.min_loss + self.eps).log() - (curr - self.min_loss + self.eps).log()
        with torch.enable_grad():
            d = torch.autograd.grad(F.softmax(self.w, -1), self.w, grad_outputs=delta.detach())[0]
        self.opt.zero_grad()
        self.w.grad = d
        self.opt.step()

def train_mtl(model, loss_fn, n_train, T, method, pref, epochs, lr=CFG['LR'], bs=CFG['BS']):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = epochs * (n_train // bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps)
    shared = list(model.backbone.parameters())
    pref_t = torch.tensor(pref, device=dev, dtype=torch.float32) * T
    famo = FAMO(T, pref) if method == 'famo' else None
    
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n_train, device=dev)
        for s in range(0, n_train - bs + 1, bs):
            idx = perm[s:s+bs]
            losses = loss_fn(model, idx)
            opt.zero_grad()
            if method == 'ls':
                sum(pref_t[i] * losses[i] for i in range(T)).backward()
            elif method == 'famo':
                famo.weighted(torch.stack(losses)).backward()
            else:
                wl = [pref_t[i] * losses[i] for i in range(T)]
                G = flat_grads(wl, shared)
                sum(wl).backward()
                g = pcgrad(G) if method == 'pcgrad' else (mgda_weights(G) @ G) * T
                o = 0
                for p in shared:
                    n = p.numel()
                    p.grad = g[o:o+n].view_as(p).clone()
                    o += n
            opt.step()
            sched.step()
            if famo is not None:
                with torch.no_grad():
                    new = torch.stack(loss_fn(model, idx))
                famo.update(torch.stack(losses).detach(), new)
    return model

def make_prefs(T, n_extra=0, seed=0):
    prefs = [[1.0 / T] * T]
    for k in range(T):
        prefs.append([0.6 if i == k else 0.4 / (T - 1) for i in range(T)])
    rng = np.random.RandomState(seed)
    for _ in range(n_extra): prefs.append(list(rng.dirichlet(np.ones(T))))
    return prefs

In [ ]:
def get_specialist_soft_targets(X, P, temp=CFG['T_KD']):
    n = len(X)
    soft_targets = np.zeros((n, K_MAX, N_CLASSES), dtype=np.float32)
    for k in range(K_MAX):
        crops = as_dev(crop_pad(X, P, k))
        bin_probs = []
        for cls_id in range(N_CLASSES):
            p = predict_binary(specialists[cls_id], crops, temp=temp).numpy()
            bin_probs.append(p)
        bin_probs = np.stack(bin_probs, axis=1)
        soft_dist = bin_probs / (bin_probs.sum(axis=1, keepdims=True) + 1e-8)
        soft_targets[:, k, :] = soft_dist
    return torch.from_numpy(soft_targets).to(dev)

g_gen = torch.Generator().manual_seed(123)
gen_lab_mask = (torch.rand(len(Xg_tr), generator=g_gen) < CFG['LABELED_FRAC']).to(dev)

def make_gen_loss(lam):
    def gen_loss_fn(model, idx):
        out = model(prep(Xg_tr_t[idx]))
        labeled = gen_lab_mask[idx]
        losses = []
        for k in range(K_MAX):
            valid = Yg_tr_t[idx, k] != -1
            if not valid.any():
                losses.append(out[k].sum() * 0)
                continue
            kd = F.kl_div(F.log_softmax(out[k][valid] / CFG['T_KD'], -1), soft_t[idx, k][valid], reduction='batchmean') * (CFG['T_KD'] ** 2)
            val_lab = labeled & valid
            ce = F.cross_entropy(out[k][val_lab], Yg_tr_t[idx, k][val_lab]) if val_lab.any() else out[k].sum() * 0
            losses.append(lam * kd + (1 - lam) * ce)
        return losses
    return gen_loss_fn

def gen_eval(model, split='val'):
    X, Y = (Xg_val_t, Yg_val_t) if split == 'val' else (Xg_te_t, Yg_te_t)
    probs = predict_gen(model, X)
    errs = []
    for k in range(K_MAX):
        valid = Y[:, k] != -1
        if valid.any():
            err = 1 - (probs[k][valid].argmax(1).to(dev) == Y[valid, k]).float().mean().item()
        else:
            err = 0.0
        errs.append(err)
    return errs

def gen_run(method, pref, lam=None, eval_test=False):
    set_seed(CFG['SEED'])
    m = MTLNet('resnet50', n_heads=K_MAX).to(dev)
    train_mtl(m, make_gen_loss(CFG['LAM'] if lam is None else lam), len(Xg_tr_t), K_MAX, method, pref, CFG['GEN_EPOCHS'])
    result = dict(err=gen_eval(m, 'val'))
    if eval_test:
        result['test_err'] = gen_eval(m, 'test')
    return result

# PENTING: Panggil make_prefs menggunakan K_MAX = 4 agar menghasilkan preferensi 4-dimensi
gen_prefs = make_prefs(K_MAX, CFG['N_PREF_EXTRA'])
print('Generated 4-dimensional preferences:', gen_prefs)